In [1]:
from google.colab import drive
drive.mount('/content/drive')
#

Mounted at /content/drive


In [2]:
import zipfile
import pandas as pd

dir='/content/drive/MyDrive/TransformersCode/e_commerce/CustomerFeedbackAnalysis'
with zipfile.ZipFile('/content/drive/MyDrive/TransformersCode/e_commerce/CustomerFeedbackAnalysis/ProductsReviews.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/drive/MyDrive/TransformersCode/e_commerce/CustomerFeedbackAnalysis')




In [3]:
df=pd.read_csv('/content/drive/MyDrive/TransformersCode/e_commerce/CustomerFeedbackAnalysis/ProductsReviews.csv')
df

,name,brand,categories,primaryCategories,reviews.date,reviews.text,reviews.title,sentiment
0,"All-New Fire HD 8 Tablet, 8"" HD Display, Wi-Fi...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",Electronics,2016-12-26T00:00:00.000Z,Purchased on Black FridayPros - Great Price (e...,Powerful tablet,Positive
1,Amazon - Echo Plus w/ Built-In Hub - Silver,Amazon,"Amazon Echo,Smart Home,Networking,Home & Tools...","Electronics,Hardware",2018-01-17T00:00:00.000Z,I purchased two Amazon in Echo Plus and two do...,Amazon Echo Plus AWESOME,Positive
2,Amazon Echo Show Alexa-enabled Bluetooth Speak...,Amazon,"Amazon Echo,Virtual Assistant Speakers,Electro...","Electronics,Hardware",2017-12-20T00:00:00.000Z,Just an average Alexa option. Does show a few ...,Average,Neutral
3,"Fire HD 10 Tablet, 10.1 HD Display, Wi-Fi, 16 ...",Amazon,"eBook Readers,Fire Tablets,Electronics Feature...","Office Supplies,Electronics",2017-08-04T00:00:00.000Z,"very good product. Exactly what I wanted, and ...",Greattttttt,Positive
4,"Brand New Amazon Kindle Fire 16gb 7"" Ips Displ...",Amazon,"Computers/Tablets & Networking,Tablets & eBook...",Electronics,2017-01-23T00:00:00.000Z,This is the 3rd one I've purchased. I've bough...,Very durable!,Positive
...,...,...,...,...,...,...,...,...
3995,Amazon - Echo Plus w/ Built-In Hub - Silver,Amazon,"Amazon Echo,Smart Home,Networking,Home & Tools...","Electronics,Hardware",2017-12-08T00:00:00.000Z,"It‚Äôs fun for the family to play with, but it...",Fun toy,Positive
3996,"Amazon Kindle E-Reader 6"" Wifi (8th Generation...",Amazon,"Computers,Electronics Features,Tablets,Electro...",Electronics,2017-03-31T00:00:00.000Z,"I love the Kindle, it is a great product. It r...",Great Product,Positive
3997,Amazon Tap - Alexa-Enabled Portable Bluetooth ...,Amazon,"Amazon Echo,Home Theater & Audio,MP3 MP4 Playe...",Electronics,2017-01-19T00:00:00.000Z,I was looking for a blutooth speaker to use wi...,"Great ""dumb"" speaker",Positive
3998,"Brand New Amazon Kindle Fire 16gb 7"" Ips Displ...",Amazon,"Computers/Tablets & Networking,Tablets & eBook...",Electronics,2016-05-27T00:00:00.000Z,This is the second Amazon Fire 7 tablet I have...,A great 7 inch tablet at a great price,Positive


In [4]:
df.columns

Index(['name', 'brand', 'categories', 'primaryCategories', 'reviews.date',
       'reviews.text', 'reviews.title', 'sentiment'],
      dtype='object')

In [9]:
from transformers import pipeline

# تحميل نموذج RoBERTa المخصص لتحليل المشاعر
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
import pandas as pd

df_clean = df.dropna(subset=['sentiment']).copy()
sentiment_counts = df_clean['sentiment'].value_counts()
min_count = sentiment_counts.min()
print(f"Min count: {min_count}\n")

balanced_df = df_clean.groupby('sentiment', group_keys=False).apply(
    lambda x: x.sample(n=min_count, random_state=42)
).reset_index(drop=True)

display(balanced_df['sentiment'].value_counts())

الفئة الأقل تكراراً هي الممثلة بـ 93 صفاً.

توزيع المشاعر بعد عملية الموازنة:


/tmp/ipykernel_411/3802927032.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df_clean.groupby('sentiment', group_keys=False).apply(


,count
sentiment,
Negative,93
Neutral,93
Positive,93


In [11]:
def predict_sentiment(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return "Neutral"

    result = sentiment_pipeline(text[:512])[0]
    label = result['label']

    label = label.lower()
    if 'positive' in label:
        return 'Positive'
    elif 'negative' in label:
        return 'Negative'
    else:
        return 'Neutral'

sample_texts = balanced_df.sample(5, random_state=42)
for idx, row in sample_texts.iterrows():
    text = row['reviews.text']
    true_sentiment = row['sentiment']
    predicted = predict_sentiment(text)

    print(f"Text: {text[:100]}...")
    print(f"True: {true_sentiment} | Pred: {predicted}")
    print("-" * 50)

النص: Has a very good Bluetooth speakers sound quality is good but otherwise she's pretty useless when it ...
التقييم الفعلي: Negative | التقييم المتوقع: Positive
--------------------------------------------------
النص: the first worked fine for three weeks then stopped. took back and got a replacement. this one lasted...
التقييم الفعلي: Neutral | التقييم المتوقع: Negative
--------------------------------------------------
النص: Love my kindle ! I enjoy reading everyday and I can easily throw in my bag with out it adding any we...
التقييم الفعلي: Positive | التقييم المتوقع: Positive
--------------------------------------------------
النص: These are great tablets for the kids and the price is wonderful with the warranty and the soft, yet ...
التقييم الفعلي: Neutral | التقييم المتوقع: Positive
--------------------------------------------------
النص: I bought this as a gift and it has performed perfectly. Can't beat at this price....
التقييم الفعلي: Positive | التقييم المتوقع: Positive
--

In [12]:
def evaluate_model_on_dataset(dataframe):
    total_samples = len(dataframe)
    correct_predictions = 0

    for idx, row in dataframe.iterrows():
        text = row['reviews.text']
        true_sentiment = row['sentiment']
        predicted = predict_sentiment(text)

        if predicted == true_sentiment:
            correct_predictions += 1

    accuracy = (correct_predictions / total_samples) * 100

    print(f"Total: {total_samples}")
    print(f"Correct: {correct_predictions}")
    print(f"Accuracy: {accuracy:.2f}%")

evaluate_model_on_dataset(balanced_df)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


جاري تشغيل النموذج وتقييم البيانات...

== تقرير التقييم ==
إجمالي عدد البيانات: 279
عدد التوقعات الصحيحة: 174
نسبة الدقة الكلية: 62.37%


### تقسيم البيانات وتجهيزها لتدريب النموذج (80% تدريب و 20% اختبار)

In [13]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

label_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
balanced_df['label'] = balanced_df['sentiment'].map(label_map)

train_df, test_df = train_test_split(balanced_df, test_size=0.2, random_state=42, stratify=balanced_df['label'])

train_dataset = Dataset.from_pandas(train_df[['reviews.text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['reviews.text', 'label']])

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['reviews.text'], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

### تهيئة النموذج وإعداد المعاملات والبدء بالتدريب

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, ignore_mismatched_sizes=True)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

trainer.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.577167,0.889671,0.660714
2,0.437136,0.834449,0.660714
3,0.198953,0.914343,0.660714


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=84, training_loss=0.4318732477369763, metrics={'train_runtime': 65.6344, 'train_samples_per_second': 10.193, 'train_steps_per_second': 1.28, 'total_flos': 44005719115008.0, 'train_loss': 0.4318732477369763, 'epoch': 3.0})

In [17]:
eval_results = trainer.evaluate()
print(f"Accuracy: {eval_results['eval_accuracy'] * 100:.2f}%")
print(f"Eval Loss: {eval_results['eval_loss']:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy
0.198953,0.889671,3,0.660714



== نتائج تقييم النموذج بعد التدريب ==
الدقة على بيانات الاختبار (Accuracy): 66.07%
قيمة الفقد في الاختبار (Eval Loss): 0.8897


In [18]:
save_directory = '/content/drive/MyDrive/TransformersCode/e_commerce/CustomerFeedbackAnalysis'
trainer.model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

تم حفظ النموذج والمجزئ بنجاح في المجلد: /content/drive/MyDrive/TransformersCode/e_commerce/CustomerFeedbackAnalysis
